In [106]:
import sklearn.datasets as datasets
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, explained_variance_score

# Testing scaling and pipelines after inital modeling.
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

Loading Dataset/Generating Lables.

In [107]:
# load diabetes dataset from sklearn.datasets import load_diabetes
dia = datasets.load_diabetes()
X = dia.data
y = dia.target # Outliners could exist within the y.target, so could adjust by cliping extreme values since MSE is penalizing itself by the large errors/outliners.

Split Dataset into 80% Training/20% Testing.

In [108]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Create Linear regression, Random Forest Regressor, and Support Vector Regression (SVR) models.

In [109]:
# linear Regression
linreg = LinearRegression()
linreg.fit(x_train, y_train)

LinearRegression()

In [110]:
# Random Forest Regressor
forest = RandomForestRegressor(n_estimators=150, random_state=42)
forest.fit(x_train, y_train) # Mess with the n_estimator to see if we can improve the accuracy of the forest model by increasing the number of trees.

RandomForestRegressor(n_estimators=150, random_state=42)

In [111]:
# Support Vector Regressor
svr = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=.1)
svr.fit(x_train, y_train)

SVR(C=100, gamma=0.1)

Models Results

In [112]:
# Predictions for each model

y_pred_linreg = linreg.predict(x_test)
y_pred_forest = forest.predict(x_test)
y_pred_svr = svr.predict(x_test)

In [113]:
# Explained Variance Score

print("Explained variance (Linear): %.2f" % explained_variance_score(y_test, y_pred_linreg))
print("Explained variance (Forest): %.2f" % explained_variance_score(y_test, y_pred_forest))
print("Explained variance (SVR): %.2f" % explained_variance_score(y_test, y_pred_svr))

# Linear regression has the highest score indicating wide dispersion.

Explained variance (Linear): 0.46
Explained variance (Forest): 0.43
Explained variance (SVR): 0.29


In [114]:
# Mean Squared Error Score

print("Mean Squared Error for Linear Regression: %.2f" % mean_squared_error(y_test, y_pred_linreg))
print("Mean Squared Error for Random Forest Regressor: %.2f" % mean_squared_error(y_test, y_pred_forest))
print("Mean Squared Error for Support Vector Regressor: %.2f" % mean_squared_error(y_test, y_pred_svr))

# Linear regression has the lowest mean squared error score indicating that it is the best fit for the data.

Mean Squared Error for Linear Regression: 2900.19
Mean Squared Error for Random Forest Regressor: 2994.67
Mean Squared Error for Support Vector Regressor: 3784.29


In [115]:
# R^2 Score
print("R^2 Score for Linear Regression: %.2f" % r2_score(y_test, y_pred_linreg))
print("R^2 Score for Random Forest Regressor: %.2f" % r2_score(y_test, y_pred_forest))
print("R^2 Score for Support Vector Regressor: %.2f" % r2_score(y_test, y_pred_svr))

# Linear regression having the highest score once more shows that it has the most variability within the variables.

R^2 Score for Linear Regression: 0.45
R^2 Score for Random Forest Regressor: 0.43
R^2 Score for Support Vector Regressor: 0.29


Best Performing Model:

For the diebeties dataset linear regression appears to do the most effective work within each catagory. Showing that it has the most wide, is the best fit of the dataset, and most variability. While forest regressor was improved by messing with the number of trees (n_estimators = 150) is getting quite closes to linear regression. 

So in order to see if I could make any improvements to the models I decided to uses k-fold cross-validation to test weather or not we could see if one of the other models aside from regression could benifit from the 

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
import numpy as np

# K-Fold Cross-Validation (k=5) WITH StandardScaler for all 3 models
# Pipeline ensures the scaler is fit only on training folds (no data leakage)

scaled_linreg = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
scaled_forest  = Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(n_estimators=150, random_state=42))])
scaled_svr     = Pipeline([('scaler', StandardScaler()), ('model', SVR(kernel='rbf', C=100, gamma=0.1, epsilon=.1))])
# Where the pipeline is created by first scaling the data and then applying the model. This ensures that the scaling is done only on the training data during each fold of cross-validation, preventing data leakage and providing a more accurate assessment of model performance.

cv_scaled_linreg = cross_val_score(scaled_linreg, X, y, cv=5, scoring='r2')
cv_scaled_forest  = cross_val_score(scaled_forest,  X, y, cv=5, scoring='r2')
cv_scaled_svr     = cross_val_score(scaled_svr,     X, y, cv=5, scoring='r2')
# The cross_val_score function is used to perform 5-fold cross-validation on each of the scaled models, evaluating their performance using the R² scoring metric. The resulting scores for each fold are stored in cv_scaled_linreg, cv_scaled_forest, and cv_scaled_svr respectively.

# Lastly we print the cross-validation R² scores for each model with scaling, along with the mean score, and then compare the improvement in mean R² scores against the unscaled versions of the models to see if scaling had a positive impact on performance.
print('--- With Scaling ---')
print('Linear Regression   — CV R² scores:', np.round(cv_scaled_linreg, 2), ' | Mean: %.2f' % cv_scaled_linreg.mean())
print('Random Forest       — CV R² scores:', np.round(cv_scaled_forest,  2), ' | Mean: %.2f' % cv_scaled_forest.mean())
print('SVR                 — CV R² scores:', np.round(cv_scaled_svr,     2), ' | Mean: %.2f' % cv_scaled_svr.mean())

print('\n--- Improvement vs Unscaled ---')
print('Linear Regression   Δ Mean R²: %+.2f' % (cv_scaled_linreg.mean() - cv_linreg.mean()))
print('Random Forest       Δ Mean R²: %+.2f' % (cv_scaled_forest.mean()  - cv_forest.mean()))
print('SVR                 Δ Mean R²: %+.2f' % (cv_scaled_svr.mean()     - cv_svr.mean()))

--- With Scaling ---
Linear Regression   — CV R² scores: [0.43 0.52 0.48 0.43 0.55]  | Mean: 0.48
Random Forest       — CV R² scores: [0.38 0.52 0.43 0.36 0.42]  | Mean: 0.42
SVR                 — CV R² scores: [0.3  0.57 0.42 0.35 0.49]  | Mean: 0.43

--- Improvement vs Unscaled ---
Linear Regression   Δ Mean R²: +0.00
Random Forest       Δ Mean R²: +0.00
SVR                 Δ Mean R²: +0.18


From the scaling only SVR improved by +0.18 making it greater than random forest, and still behind linear regression. 
So SVR was being harmed by not being scaled, the other models (linear regression, and random forest), 
were unaffected since linear was scale-invariant (cahnges in unit mesurements does not affect outcome) and forest split on thresholds (numerical values used at each internal node of every decision tree to partition the data into two branches).

After some more testing it still appears that linear regression still takes the lead and while I almost closed the gap with the other 2 models, neither got past.